In [1]:
import sys
sys.path.append('..')

In [2]:
from splice import _LocWrapper

In [3]:
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
SPLICE_NAME = "climplicit_test_2000"
DEVICE      = "cuda"

In [5]:
from pathlib import Path
from paths import load_paths

_paths   = load_paths()
SPLICE_MODEL = _paths["splice"][SPLICE_NAME]["model"]
LAT_LON_CSV  = "/home/libe2152/data/dense_grid/dense_grid.csv"
OUTPUT       = Path("reports") / f"splice_{SPLICE_NAME}.json"
print("Splice model:", SPLICE_MODEL)
print("Lat/lon CSV: ", LAT_LON_CSV)

Splice model: /home/libe2152/projects/explainable-earth-embeddings/splice_results/txt-open_clip_vit_l__loc-csp_fmow__tproj-linear__lproj-none__tft-lora__lft-only_proj__lora_r-8__loss-clip_symmetric__lr-0.0001__h-86357a7e/test_concepts_nc_15000_top_100000/splice_model.pt
Lat/lon CSV:  /home/libe2152/data/dense_grid/dense_grid.csv


In [6]:
NUM_SAMPLES = 200

In [7]:
from eval_splice import evaluate

report = evaluate(SPLICE_MODEL, LAT_LON_CSV, DEVICE, str(OUTPUT), num_samples=NUM_SAMPLES)

/home/libe2152/miniconda3/envs/fai/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


SpLiCE solver = admm
SpLiCE location mean embedding norm: 0.44542941451072693


Iterating through latlons...: 100%|██████████| 1/1 [00:02<00:00,  2.71s/it]

Stopping at iteration 800
Prime Residual, r_k: 1.1367873042900101e-07
Dual Residual, s_k: 1.453012288266109e-07
Saved sparse embeddings to /home/libe2152/data/splice_embeddings/txt-open_clip_vit_l__loc-csp_fmow__tproj-linear__lproj-none__tft-lora__lft-only_proj__lora_r-8__loss-clip_symmetric__lr-0.0001__h-86357a7e__test_concepts_nc_15000_top_100000__dense_grid__n200_s0.pt


MSE: 0.002509
R2:  -2702975.250000
Saved to reports/splice_csp_fmow_test_2000.json


In [8]:
import json
report = json.loads(OUTPUT.read_text())
for k, v in report.items():
    print(f"{k:20s} {v}")

splice_model         /home/libe2152/projects/explainable-earth-embeddings/splice_results/txt-open_clip_vit_l__loc-csp_fmow__tproj-linear__lproj-none__tft-lora__lft-only_proj__lora_r-8__loss-clip_symmetric__lr-0.0001__h-86357a7e/test_concepts_nc_15000_top_100000/splice_model.pt
lat_lon_csv          /home/libe2152/data/dense_grid/dense_grid.csv
n_samples            200
mse                  0.0025090351700782776
r2                   -2702975.25
cosine_sim_min       0.11852535605430603
cosine_sim_max       0.8005886077880859
cosine_sim_mean      0.6788435578346252
avg_active_concepts  80.19499969482422
frac_all_zero_solutions 0.0


In [9]:
import torch, torch.nn.functional as F
from eval_splice import _sparse_emb_path

splice = torch.load(SPLICE_MODEL, map_location=DEVICE, weights_only=False)
splice.eval()

cache_path = _sparse_emb_path(SPLICE_MODEL, LAT_LON_CSV, num_samples=NUM_SAMPLES)
cached   = torch.load(cache_path, map_location="cpu", weights_only=True)
orig_t   = cached["orig"]          # (N, D) — L2-normalized location embeddings
sparse_t = cached["sparse"]        # (N, C) — LASSO weights
with torch.no_grad():
    recon_t = splice.recompose_image(sparse_t.to(DEVICE)).cpu()


In [10]:
print(f"image_mean norm: {splice.image_mean.norm():.4f}")
# Near 1 = embeddings are concentrated; near 0 = well spread


image_mean norm: 0.4454


In [11]:
active = (sparse_t > 0).float().sum(dim=1)
all_zero = (sparse_t.sum(dim=1) == 0).float().mean()
print(f"avg active concepts per location: {active.mean():.2f}")
print(f"fraction with ALL-ZERO weights:   {all_zero:.4f}")


avg active concepts per location: 80.19
fraction with ALL-ZERO weights:   0.0000


In [12]:
centered = F.normalize(orig_t - splice.image_mean.cpu(), dim=1)  # how SPLICE sees the input
dict_norm = F.normalize(splice.dictionary.cpu(), dim=1)           # (C, D)
sims = centered @ dict_norm.T                                      # (N, C)
print(f"max sim to any concept — mean: {sims.max(dim=1).values.mean():.4f}, "
      f"min: {sims.max(dim=1).values.min():.4f}")
# If mean max-sim is low (< 0.1), the dictionary can't express the location embeddings


max sim to any concept — mean: 0.1779, min: 0.1255


In [13]:
# Are location and concept embeddings in the same space at all?
loc_vs_concepts = orig_t @ dict_norm.T                 # (N, C), before centering
print(f"loc↔concept cosine: mean={loc_vs_concepts.mean():.4f}, "
      f"std={loc_vs_concepts.std():.4f}, "
      f"max={loc_vs_concepts.max():.4f}")
# Low / near-zero mean + tiny std = spaces are not aligned


loc↔concept cosine: mean=-0.0002, std=0.0507, max=0.2635


In [14]:
from external.splice.splice.model import SPLICE

orig_t = orig_t.to("cuda:0")
for alpha in [1e-16, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 0.001, 0.01, 0.05, 0.1, 0.25, 0.35, 0.5, 1.0]:
    test = SPLICE(splice.image_mean, splice.dictionary,
                  solver='admm', l1_penalty=alpha,
                  return_weights=True, device="cuda:0")
    with torch.no_grad():
        w = test.decompose(F.normalize(orig_t[:200], dim=1))
        r = test.recompose_image(w)
    cos = (r * orig_t[:200]).sum(dim=1).mean()
    zeros = (w.sum(dim=1) == 0).float().mean()
    r2 = r2_score(orig_t[:200].cpu(), r.cpu(), multioutput="uniform_average")
    print(f"l1={alpha:.3f}  cos_sim={cos:.4f}  all-zero-frac={zeros:.4f}  r2-score={r2:.4f}")


Using ADMM solver...
With rho value 5.0
Stopping at iteration 1999
Prime Residual, r_k: 2.481109049767838e-06
Dual Residual, s_k: 0.00019344667089171708
l1=0.000  cos_sim=0.9551  all-zero-frac=0.0000  r2-score=0.8848
Using ADMM solver...
With rho value 5.0
Stopping at iteration 1999
Prime Residual, r_k: 2.4889786800486036e-06
Dual Residual, s_k: 0.00019358732970431447
l1=0.000  cos_sim=0.9551  all-zero-frac=0.0000  r2-score=0.8853
Using ADMM solver...
With rho value 5.0
Stopping at iteration 1999
Prime Residual, r_k: 2.491868144716136e-06
Dual Residual, s_k: 0.00019356216944288462
l1=0.000  cos_sim=0.9551  all-zero-frac=0.0000  r2-score=0.8842
Using ADMM solver...
With rho value 5.0
Stopping at iteration 1999
Prime Residual, r_k: 2.9116795303707477e-06
Dual Residual, s_k: 0.00019795064872596413
l1=0.000  cos_sim=0.9551  all-zero-frac=0.0000  r2-score=0.8833
Using ADMM solver...
With rho value 5.0
Stopping at iteration 1999
Prime Residual, r_k: 5.176359991310164e-06
Dual Residual, s_k: 